# Model Dev — train & test

Train a model to predict the **finishing order** of a race from the features built in `data_organize`, compare it against a grid-only baseline, and inspect its predictions race-by-race.

Run order upstream: `data_loading` -> `data_organize` -> this notebook.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.stats import spearmanr

from f1pred import paths

df = pd.read_parquet(paths.DATA / "processed" / "features.parquet")
df = df.sort_values(["season", "round"]).reset_index(drop=True)

assert "target" in df.columns and len(df) > 0
df.head()

## 1. Config & time-based split

Predicting the future -> train on older seasons, validate on newer ones. A random split would leak future races into training.

In [ ]:
FEATURES = ["grid_start", "quali_position", "quali_gap", "driver_form", "team_form", "grid_penalty"]
TARGET = "target"

train_df = df[df["season"] <= 2024].reset_index(drop=True)
valid_df = df[df["season"] >= 2025].reset_index(drop=True).copy()

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_valid, y_valid = valid_df[FEATURES], valid_df[TARGET]

assert len(train_df) > 0 and len(valid_df) > 0

## 2. Metric helpers

Both metrics score a *ranking column* (lower value = better predicted finish) against the true finishing order, per race, then average across races.

In [ ]:
def per_race_spearman(frame, rank_col):
    """Mean per-race Spearman between a ranking column and the true order."""
    scores = [
        spearmanr(g[rank_col], g[TARGET]).correlation
        for _, g in frame.groupby(["season", "round"]) if len(g) >= 2
    ]
    return float(np.mean(scores))


def top1_accuracy(frame, rank_col):
    """Share of races where the predicted winner is the actual winner."""
    hits = [
        g.loc[g[rank_col].idxmin(), "DriverId"] == g.loc[g[TARGET].idxmin(), "DriverId"]
        for _, g in frame.groupby(["season", "round"])
    ]
    return float(np.mean(hits))

assert callable(per_race_spearman) and callable(top1_accuracy)

## 3. Baseline — grid only

The simplest possible predictor: assume everyone finishes where they started. The model has to beat this to be worth anything.

In [ ]:
base_spearman = per_race_spearman(valid_df, "grid_start")
base_top1 = top1_accuracy(valid_df, "grid_start")

assert base_spearman > 0
{"baseline_spearman": round(base_spearman, 3), "baseline_top1": round(base_top1, 3)}

## 4. Train

Start simple: regress the finishing position, then sort by prediction to get the predicted order. (`LGBMRanker` / LambdaMART is the natural next upgrade.)

In [ ]:
model = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, random_state=0, verbose=-1)
model.fit(X_train, y_train)

assert model.n_features_in_ == len(FEATURES)

## 5. Evaluate vs baseline

Same metrics on the model's predicted order. Expect it to edge out the grid baseline on Spearman.

In [ ]:
valid_df["pred"] = model.predict(X_valid)   # lower pred = better predicted finish

model_spearman = per_race_spearman(valid_df, "pred")
model_top1 = top1_accuracy(valid_df, "pred")

assert model_spearman > 0
pd.DataFrame({
    "spearman": [base_spearman, model_spearman],
    "top1": [base_top1, model_top1],
}, index=["grid_baseline", "model"]).round(3)

## 6. Inspect one race

Sanity-check the predictions you can read: the most recent validation race, predicted order vs actual order.

In [ ]:
key = valid_df[["season", "round"]].drop_duplicates().iloc[-1]
one = valid_df[(valid_df["season"] == key["season"]) & (valid_df["round"] == key["round"])].copy()
one["pred_order"] = one["pred"].rank(method="first").astype(int)
one = one.sort_values("pred_order")

assert len(one) > 0
print(f"{key['season']} round {key['round']}")
one[["Abbreviation", "TeamId", "grid_start", "pred_order", "target"]]

## 7. Feature importance

Which signals the model leans on. A quick smell-test that nothing degenerate is going on.

In [ ]:
importance = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)

assert importance.sum() > 0
importance

## 8. Save the model

Persist to `results/` so a prediction notebook can load it later.

In [ ]:
import joblib

out = paths.RESULTS / "model.pkl"
joblib.dump(model, out)

assert out.exists()